In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym

In [2]:
class DQNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(DQNetwork, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )
    def forward(self, x):
        return self.fc(x)

In [3]:
class SimplePERBuffer:
    def __init__(self, capacity, alpha=0.6):
        self.capacity = capacity
        self.alpha = alpha
        self.buffer = []
        self.priorities = []

    def add(self, transition, error):
        priority = (abs(error) + 1e-5) ** self.alpha
        if len(self.buffer) < self.capacity:
            self.buffer.append(transition)
            self.priorities.append(priority)
        else:
            idx = len(self.buffer) % self.capacity
            self.buffer[idx] = transition
            self.priorities[idx] = priority

    def sample(self, batch_size):
        probs = np.array(self.priorities) / sum(self.priorities)
        indices = np.random.choice(len(self.buffer), batch_size, p=probs)
        samples = [self.buffer[idx] for idx in indices]
        return samples, indices

    def update_priorities(self, indices, errors):
        for idx, error in zip(indices, errors):
            self.priorities[idx] = (abs(error) + 1e-5) ** self.alpha

In [4]:
env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

policy_net = DQNetwork(state_dim, action_dim)
target_net = DQNetwork(state_dim, action_dim)
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)
memory = SimplePERBuffer(capacity=1000)

In [9]:
raw_state, _ = env.reset()
action = env.action_space.sample()
next_raw_state, reward, terminated, truncated, _ = env.step(action)
done = terminated or truncated

state_t = torch.FloatTensor(raw_state)
next_state_t = torch.FloatTensor(next_raw_state)

with torch.no_grad():
    current_q = policy_net(state_t)[action].item()
    max_next_q = target_net(next_state_t).max().item()
    target_q = reward + 0.99 * max_next_q * (1 - int(done))
    initial_error = target_q - current_q

memory.add((raw_state, action, reward, next_raw_state, done), initial_error)

In [8]:
samples, indices = memory.sample(batch_size=1)
s, a, r, ns, d = samples[0]

pred_q = policy_net(torch.FloatTensor(s))[a]
with torch.no_grad():
    target_q = r + 0.99 * target_net(torch.FloatTensor(ns)).max() * (1 - int(d))

loss = nn.MSELoss()(pred_q, target_q)
optimizer.zero_grad()
loss.backward()
optimizer.step()

In [7]:
new_error = abs((target_q - pred_q).item())
memory.update_priorities(indices, [new_error])

print("DQN with Prioritized Experience Replay initialized and updated successfully!")

DQN with Prioritized Experience Replay initialized and updated successfully!
